# Phase 0 + Phase 1 on a T4 (Kaggle or Colab)

**Kaggle:** Settings → Accelerator **GPU T4 x2**, Internet **On**. **Colab:** Runtime → GPU (T4).

Phase 0 checks the measurement hooks work (warm-up, host/GPU step split, provenance, token gate).
Phase 1 re-baselines every number in `docs/checkpoint.md` §1, five interleaved runs per arm.
Every result lands in `results/t4/<date>_<commit>/`. Commit or download that folder **before the session ends**.

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,clocks.max.sm,power.limit --format=csv
import os
ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
print("kaggle" if ON_KAGGLE else "colab")

Clone the branch that has the Phase 0 hooks (push it first). Pin to one GPU: both T4s share one host CPU, and the host staging path is part of what is being measured.

In [ ]:
os.chdir("/kaggle/working" if ON_KAGGLE else "/content")
!git clone -b t4-phase0 https://github.com/Vaibhav7711/full-inference-engine.git fol
%cd fol
!git log -1 --oneline
%env CUDA_VISIBLE_DEVICES=0
%env TOKENIZERS_PARALLELISM=false

In [ ]:
!bash scripts/setup_kaggle.sh   # same script on Colab: never reinstalls torch
!python scripts/colab_preflight.py

Results folder keyed by date and commit, so a JSON can always be tied to the code that produced it.

In [ ]:
import subprocess, datetime, pathlib
SHA = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
RUN = f"results/t4/{datetime.date.today():%Y%m%d}_{SHA}"
pathlib.Path(RUN).mkdir(parents=True, exist_ok=True)
%env RUN=$RUN
print(RUN)

## Phase 0 — gates

**0.1 Correctness on the GPU.** Includes the new strided-SwiGLU test and every engine path the CPU suite skips. Nothing below means anything if this fails.

In [ ]:
!python -m pytest -q -x -m cuda tests/kernels tests/batching tests/cache tests/reliability tests/server tests/correctness

**0.2 Hooks work.** Warm-up must capture every bucket in both decode regimes (`graphs == 2 × buckets`) and compile both prefill paths (`prefill_sdpa_calls > 0` and `prefill_chunked_calls > 0`). Then one instrumented decode step should report all four phases, and the host stage should be well under 1 ms at this batch.

In [ ]:
import time, torch
from engine.model import load_model
from engine.batching.continuous_batching import ContinuousBatchingEngine
from benchmarks.common import device_clock_record, git_record

loaded = load_model("Qwen/Qwen3-0.6B")
engine = ContinuousBatchingEngine(loaded.model, loaded.tokenizer, loaded.device,
                                  num_blocks=1024, max_active=8, cuda_graph_batch_sizes=(2, 4, 8))
t0 = time.perf_counter(); summary = engine.warmup(); warm_s = time.perf_counter() - t0
print(f"warmup {warm_s:.1f}s ->", summary)
assert summary["graphs"] == 2 * 3, summary
assert summary["prefill_sdpa_calls"] and summary["prefill_chunked_calls"], summary

engine.instrument = True
prompts = ["Explain KV caching in one sentence."] * 8
out = engine.generate(prompts, max_new_tokens=8)
print("last step phases (ms):", {k: round(v, 3) for k, v in engine.last_step_timing.items()})
assert {"host_stage_ms", "decode_gpu_ms", "sync_ms"} <= set(engine.last_step_timing)
print("git:", git_record()); print("clocks:", device_clock_record())
del engine, loaded; torch.cuda.empty_cache()

**0.3 Token identity across arms + interleaving.** A short A/B on the cheapest setting proves the gate runs and the arms are ordered A B A B. Two runs only — this is a plumbing check, not a result.

In [ ]:
!python -m benchmarks.reliability.ab --setting cuda_graphs --repeats 2 --duration 4 \
    --out $RUN/phase0_smoke_cuda_graphs.json

## Phase 1 — re-baseline

Order matters: the roofline gives the floor everything else is compared to; the profile shows where the post-fix decode step goes before any A/B is interpreted.
Each A/B: 5 interleaved runs per arm, `chat` profile unless the setting only binds on longer prompts. `--cuda-graphs` puts both arms on the graphed decode path so the prefill settings are the only difference.
A 20 s pause between scripts lets the card cool so the first run of each A/B is not measured on a hot GPU.

In [ ]:
!python -m benchmarks.kernels.roofline --out $RUN/roofline.json

In [ ]:
!sleep 20 && python -m benchmarks.batching.profile_continuous_decode --concurrency 8 --output $RUN/profile_continuous_decode.json

In [ ]:
!sleep 20 && python -m benchmarks.reliability.ab --setting cuda_graphs --repeats 5 --prompt-profile chat \
    --out $RUN/ab_cuda_graphs_chat.json

**Decides the `tiled_prefill` default.** If `tiled` loses on `prefill_step_p50_ms` outside the spread, flip the default to `False` in the next commit and record the retraction.

In [ ]:
!sleep 20 && python -m benchmarks.reliability.ab --setting prefill_kernel --repeats 5 --prompt-profile chat --cuda-graphs \
    --out $RUN/ab_prefill_kernel_chat.json

In [ ]:
!sleep 20 && python -m benchmarks.reliability.ab --setting prefill_chunk --repeats 5 --prompt-profile chat --cuda-graphs \
    --out $RUN/ab_prefill_chunk_chat.json

In [ ]:
!sleep 20 && python -m benchmarks.reliability.ab --setting prefix_cache --repeats 5 --prompt-profile chat --cuda-graphs \
    --out $RUN/ab_prefix_cache_chat.json

INT8 KV only matters when KV reads are a large share of step bytes, i.e. long contexts. Token drift is expected for this setting and the gate reports it as a note.

In [ ]:
!sleep 20 && python -m benchmarks.reliability.ab --setting kv_dtype --repeats 5 --prompt-profile long --cuda-graphs \
    --num-blocks 512 --out $RUN/ab_kv_dtype_long.json

Phase 9 re-run: the fused gate/up projection no longer pays two activation copies, so its earlier 'neutral' verdict was measuring the copies.

In [ ]:
!sleep 20 && python -m benchmarks.batching.mlp_gate_up_fusion_ab --rounds 7 --output $RUN/mlp_gate_up_fusion_ab.json

Tile sweep for the tiled prefill kernel against the dense-SDPA upper bound. Whatever the best variant is, it must beat `per_token` here *and* in the engine A/B above to earn the default.

In [ ]:
!sleep 20 && python -m benchmarks.kernels.prefill_attention_ab --sweep-tiles --out $RUN/prefill_attention_sweep.json

## Summary table

One line per A/B: the verdict on the metrics that decide things. `unresolved` means inside run-to-run spread — record it as unresolved, not as a small win.

In [ ]:
import json, glob, os
rows = []
for path in sorted(glob.glob(f"{RUN}/ab_*.json")):
    d = json.load(open(path))
    labels = list(d["arms"])
    print(f"\n{os.path.basename(path)}  [{labels[1]} vs {labels[0]}]  "
          f"tokens identical={d['token_identity']['identical']}  "
          f"clocks before/after SM MHz: {d['clocks_before'].get('clocks.sm')} / {d['clocks_after'].get('clocks.sm')}")
    for metric in ("step_timing.expected_gap_ms", "step_timing.decode_step_p50_ms",
                   "step_timing.prefill_step_p50_ms", "latency.ttft_p50", "latency.itl_p99",
                   "step_timing.host_stage_ms_p50", "step_timing.decode_gpu_ms_p50"):
        base = d["arms"][labels[0]]["summary"].get(metric, {})
        print(f"  {metric:36s} base={base.get('median', float('nan')):8.3f}  {d['comparison'].get(metric, '-')}")

## Save before the session dies

Either push from here (set a token) or zip and download. Both keep the folder name, which carries the commit.

In [ ]:
!git add $RUN && git -c user.name=t4-runner -c user.email=t4@local commit -m "T4 phase 0/1 results $RUN" && git log -1 --oneline
# !git push https://<TOKEN>@github.com/Vaibhav7711/full-inference-engine.git HEAD:main
!zip -qr /kaggle/working/t4_results.zip $RUN 2>/dev/null || zip -qr /content/t4_results.zip $RUN
!ls -la $RUN